# 📊 Prompt Engineering Benchmark: Human vs. LLM-Optimized Strategies
**Does Prompt Structure Affect LLM Price Prediction? A Controlled Experiment**

_Week 7 Community Exercise · LiteLLM · HuggingFace · Gradio · OpenRouter_

---

## What I Learned in Week 7

Week 7 focused on fine-tuning and evaluating open-source models. A key insight was the dramatic impact of **prompt format** on model performance:
- **Day 2** — `item.make_prompts(tokenizer, CUTOFF, include_label)` showed that token structure and wording are critical for fine-tuning.
- **Day 5** — Evaluation revealed that the same model weights produce different accuracy (MAE) scores based on the query structure.

## The Experiment

This project empirically validates these findings by comparing 8 different prompt strategies on the same 20 test items from `ed-donner/items_lite`:
1. **4 Human-Crafted Prompts**: Minimal, Structured, Chain-of-Thought, and Persona-based.
2. **3 LLM-Generated Prompts**: Created using **Meta-Prompting** (tasking an LLM to engineer its own optimal prompts).
3. **1 Super Prompt**: An LLM-optimized combination of the best elements from all 7 strategies.

We use **Mean Absolute Error (MAE)** to rank these formats: the average dollar difference between the LLM's prediction and the actual price.

---

## Components

| Component | Purpose |
|---|---|
| `ed-donner/items_lite` | Amazon product dataset (HuggingFace Hub) |
| `litellm` + OpenRouter | Unified API access to frontier models (e.g., GPT-5 Nano) |
| Meta-Prompting | Asking an LLM to generate optimal system and user prompts |
| Benchmark Engine | Automated ranking of 8 formats using MAE score |
| Gradio UI | Interactive results dashboard + Live "Try Your Own Prompt" tester |

---

## How It Works

1. **Load** — Fetch `test` split from HuggingFace.
2. **Generate** — Use Meta-Prompting to create 3 LLM-optimized strategies and 1 final Super Prompt.
3. **Benchmark** — Run all 8 formats, calculate MAE vs. actual prices, and rank by performance.
4. **Explore** — Compare rankings via Gradio and test your own custom templates in real-time.


## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation if needed
!uv pip install -q litellm gradio datasets python-dotenv

print("✅ Dependencies installed.")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import random
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(override=True)

# ── API / Model config ───────────────────────────────────────────────────────
#
#  Default: Ollama running locally (free, no API key)
#  To use a fine-tuned HF model from Week 7, change MODEL_ID to:
#      "huggingface/<your-username>/<your-model>"
#  Or switch to a frontier model:
#      "openai/gpt-4.1-nano"  →  requires OPENAI_API_KEY in .env
#      "openrouter/openai/gpt-4.1-nano"  →  requires OPENROUTER_API_KEY
#
# MODEL_ID       = "ollama/llama3.2"     # ← change this to your model
OLLAMA_BASE    = "http://localhost:11434"  # only used when MODEL_ID starts with "ollama/"
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

MODEL_ID = "openrouter/openai/gpt-4.1-nano"
if MODEL_ID.startswith("ollama/"):
    MODEL_API_BASE = OLLAMA_BASE
    MODEL_API_KEY = None
else:
    MODEL_API_BASE = OPENROUTER_BASE
    MODEL_API_KEY = OPENROUTER_API_KEY

from litellm import completion

class LLMClient:
    def __init__(self, model_id: str, api_base: str, api_key=None):
        self.model_id = model_id
        self.api_base = api_base
        self.api_key = api_key

    def complete(self, system_content: str, user_content: str, max_tokens: int = 80) -> str:
        print(f"Sending request to model {self.model_id} at {self.api_base}...")
        if self.api_key is not None:
            print(f"Using API key: {self.api_key[:8]}...")
        else:
            print("No API key provided.")
        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ]
        kwargs = dict(model=self.model_id, messages=messages, max_tokens=max_tokens, api_base=self.api_base)
        if self.api_key is not None:
            kwargs["api_key"] = self.api_key
        
        resp = completion(**kwargs)
        content = resp.choices[0].message.content
        print(f"Content: {content}")

        return content if content is not None else ""
        

llm_client = LLMClient(MODEL_ID, MODEL_API_BASE, MODEL_API_KEY)

BENCHMARK_SIZE = 20    # items to evaluate per format
RANDOM_SEED    = 42
random.seed(RANDOM_SEED)

# HuggingFace login (needed for dataset loading)
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print(f"HuggingFace token: {hf_token[:8]}...")
else:
    print("⚠️  No HF_TOKEN — dataset loading will attempt anonymous access")

print(f"Model: {MODEL_ID}")
print(f"Benchmark size: {BENCHMARK_SIZE} items per format")
print("\n✅ Configuration ready.")

## Cell 3 — Load Dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

print("Loading ed-donner/items_lite from HuggingFace Hub...")

raw = load_dataset("ed-donner/items_lite")

test_data = [
    {"summary": r["summary"], "price": float(r["price"])}
    for r in raw["test"]
]

print(f"Test items loaded: {len(test_data):,}")
print(f"\nExample item:")
print(f"  Summary: {test_data[0]['summary'][:150]}...")
print(f"  Price  : ${test_data[0]['price']:.2f}")

# Fix benchmark subset
random.seed(RANDOM_SEED)
bench_items = random.sample(test_data, min(BENCHMARK_SIZE, len(test_data)))
print(f"\nBenchmark subset: {len(bench_items)} items (fixed seed={RANDOM_SEED})")

## Cell 4 — Define 4 Human Prompt Formats

In [ ]:
# ── Format A: Minimal ───────────────────────────────────────────────────────
# One-liner instruction. As close to the raw text as possible.
# Mirrors how many fine-tuning datasets look when no system prompt is used.

PROMPT_A_SYSTEM = "Price this product. Reply with the dollar amount only."

def format_a(item):
    return PROMPT_A_SYSTEM, item["summary"]


# ── Format B: Structured ────────────────────────────────────────────────────
# Mirrors the Week 7 Day 2 prompt pattern: explicit role, rules, and output format.

PROMPT_B_SYSTEM = """You are a professional retail pricing analyst.
Your task is to estimate the retail price of a product based on its description.

Rules:
- Consider the product category, brand, and features
- Price must be between $1 and $1000
- Respond ONLY with the price in the format: $XX.XX
- Do NOT include any explanation"""

def format_b(item):
    user_msg = f"Product description:\n{item['summary']}\n\nEstimated price:"
    return PROMPT_B_SYSTEM, user_msg


# ── Format C: Chain-of-Thought (compressed) ─────────────────────────────────
# Asks the model to reason briefly before giving the price.
# Inspired by Week 7 Day 5 observation that reasoning prompts can improve accuracy.

PROMPT_C_SYSTEM = """You are a pricing expert. Think step by step:
1. Identify the product type and typical price range
2. Consider brand, quality signals, and features mentioned
3. Arrive at a specific price estimate

Format your response as:
Reasoning: <1-2 sentences only>
Price: $XX.XX"""

def format_c(item):
    return PROMPT_C_SYSTEM, item["summary"]


# ── Format D: Persona + constraints ─────────────────────────────────────────
# Different strategy: thrift-store buyer persona, conservative estimate.

PROMPT_D_SYSTEM = """You are a thrift-store buyer. Give a conservative retail price estimate.
Consider only clear signals: category, brand, and key features. Reply with only $XX.XX. Price between $1 and $1000."""

def format_d(item):
    user_msg = f"Product:\n{item['summary']}\n\nYour price estimate:"
    return PROMPT_D_SYSTEM, user_msg


FORMATS = {
    "A — Minimal"             : format_a,
    "B — Structured"          : format_b,
    "C — Chain-of-Thought"    : format_c,
    "D — Persona"             : format_d,
}

# Preview each format
for name, fn in FORMATS.items():
    sys_p, usr_p = fn(bench_items[0])
    print(f"\n{'='*60}")
    print(f"Format {name}")
    print(f"  System: {sys_p[:80]}...")
    print(f"  User  : {usr_p[:80]}...")

print("\n✅ 4 human prompt formats defined.")

In [ ]:
def llm_completions(system_content, user_content, max_tokens):
    """Generic LLM pricer function — calls any model via OpenRouter."""
    
    messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ]
    resp = completion(
        model=MODEL_ID,
        messages=messages,
        api_base=MODEL_API_BASE,
        api_key=MODEL_API_KEY,
    )
    content = resp.choices[0].message.content
    return (content or "").strip()

# result = llm_completions("As an AI assistant, you are", "Find the price of this product: iPhone 15 Pro Max")
# print("Result:", result)

print("LLM completions is ready")

## Cell 4b — Generate 3 prompt strategies via LLM

Use a base (meta) prompt to ask the LLM to craft 3 distinct prompt strategies. Each strategy has a system message and a user template with `{summary}` placeholder. Results are cached so re-runs skip regeneration.

In [ ]:
import json

META_SYSTEM = """You are a prompt engineer. Output valid JSON only, no markdown or extra text."""

META_USER = """Task: Given a product description (summary), the model must output a single retail price estimate.
Constraints: price between $1 and $1000. Output must be parseable (e.g. "Price: $XX.XX" or "$XX.XX").
User message template must use exactly one placeholder: {summary} (will be replaced with the product text).

Generate 3 different prompt strategies using different personas (direct, analyst, expert) and distinct approaches (e.g. role, reasoning style, format).
Return JSON in this exact shape:
{"strategies": [
  {"name": "LLM-1 direct persona", "system": "system prompt text", "user_template": "user message with {summary}"},
  {"name": "LLM-2 analyst persona", "system": "system prompt text", "user_template": "user message with {summary}"},
  {"name": "LLM-3 expert persona", "system": "system prompt text", "user_template": "user message with {summary}"}
]}"""

CACHE_PATH = "llm_generated_prompts.json"
REFRESH_LLM_PROMPTS = False

def _call_for_json(system_msg: str, user_msg: str) -> str:
    return llm_completions(system_msg, user_msg)

def _extract_json_object(text: str):
    text = (text or "").strip()
    if not text:
        return None
    if text.startswith("```"):
        text = re.sub(r"^```\w*\n?", "", text).strip()
        text = re.sub(r"\n?```\s*$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start : end + 1]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def _normalize_strategy_item(s, i):
    if not isinstance(s, dict):
        return None
    name = s.get("name") or s.get("label") or f"LLM-{i+1}"
    system = s.get("system") or s.get("system_prompt") or ""
    user_tpl = (s.get("user_template") or s.get("user_message") or s.get("user_prompt") or "").replace("{product}", "{summary}")
    if not user_tpl or "{summary}" not in user_tpl:
        user_tpl = "Product:\n{summary}\n\nPrice:"
    return {"name": name, "system": system, "user_template": user_tpl}

def _parse_llm_strategies(raw: str):
    data = _extract_json_object(raw)
    if not data:
        return {}, []
    strategies = None
    if isinstance(data, list) and len(data) > 0:
        strategies = data
    elif isinstance(data, dict):
        strategies = data.get("strategies") or data.get("prompts") or data.get("prompt_strategies")
        if not isinstance(strategies, list) or len(strategies) == 0:
            for v in data.values():
                if isinstance(v, list) and len(v) > 0 and isinstance(v[0], (dict, list)):
                    strategies = v
                    break
            if not strategies or len(strategies) == 0:
                strategies = [v for v in data.values() if isinstance(v, dict) and (v.get("system") or v.get("user_template") or v.get("name") or v.get("system_prompt"))]
    if not isinstance(strategies, list) or len(strategies) == 0:
        return {}, []
    normalized = []
    for i, s in enumerate(strategies):
        item = _normalize_strategy_item(s, i)
        if item and (item["system"] or item["user_template"]):
            normalized.append(item)
        if len(normalized) >= 3:
            break
    fallback = {"name": "Fallback", "system": "Estimate the product retail price. Reply with only the price as $XX.XX.", "user_template": "Product:\n{summary}\n\nPrice:"}
    while len(normalized) < 3:
        normalized.append({**fallback, "name": f"LLM-{len(normalized)+1} padded"})
    result = {}
    parsed_list = []
    for i, item in enumerate(normalized[:3]):
        name, system, user_tpl = item["name"], item["system"], item["user_template"]
        key = f"LLM-{i+1} — {name}" if isinstance(name, str) else f"LLM-{i+1}"
        def _make_fn(sys, tpl):
            return lambda it: (sys, tpl.replace("{summary}", it["summary"]))
        result[key] = _make_fn(system, user_tpl)
        parsed_list.append({"name": name, "system": system, "user_template": user_tpl})
    return result, parsed_list

FORMATS_LLM = {}
if not REFRESH_LLM_PROMPTS and os.path.exists(CACHE_PATH):
    try:
        with open(CACHE_PATH) as f:
            cached = json.load(f)
        strategies_list = cached.get("strategies", [])
        for i, s in enumerate(strategies_list[:3]):
            name = s.get("name", f"LLM-{i+1}")
            key = f"LLM-{i+1} — {name}"
            sys_p, tpl = s.get("system", ""), (s.get("user_template") or "").replace("{product}", "{summary}")
            if "{summary}" not in tpl:
                tpl = "Product:\n{summary}\n\nPrice:"
            FORMATS_LLM[key] = (lambda sys, t: lambda item: (sys, t.replace("{summary}", item["summary"])))(sys_p, tpl)
        print(f"Loaded 3 LLM strategies from {CACHE_PATH}")
    except Exception as e:
        print(f"Cache read failed: {e}. Regenerating.")
        FORMATS_LLM = {}

if not FORMATS_LLM:
    print("Calling LLM to generate 3 prompt strategies...")
    try:
        raw = _call_for_json(META_SYSTEM, META_USER)
        FORMATS_LLM, parsed_list = _parse_llm_strategies(raw)
        if FORMATS_LLM and parsed_list:
            with open(CACHE_PATH, "w") as f:
                json.dump({"strategies": parsed_list}, f, indent=2)
            print(f"Cached to {CACHE_PATH}")
        else:
            raise ValueError("Parse returned no strategies")
    except Exception as e:
        print(f"LLM generation failed: {e}. Using fallbacks.")
        fallback_sys = "Estimate the product retail price. Reply with only the price as $XX.XX."
        FORMATS_LLM = {
            "LLM-1 — Fallback direct": lambda item: (fallback_sys, f"Product:\n{item['summary']}\n\nPrice:"),
            "LLM-2 — Fallback analyst": lambda item: ("You are a pricing analyst. Output price only.", f"{item['summary']}\n\nEstimated price:"),
            "LLM-3 — Fallback concise": lambda item: ("Price this product. Reply $XX.XX only.", item["summary"]),
        }

print(f"FORMATS_LLM: {list(FORMATS_LLM.keys())}")

## Cell 4c — Generate super prompt from all 7

Combine the 4 human + 3 LLM prompt strategies into one meta-prompt; ask the LLM to produce a single "super" (system + user template) that incorporates the best of all seven. The super prompt is then benchmarked as the 8th format.

In [ ]:
all_seven = dict(**FORMATS, **FORMATS_LLM)
seven_text = []
for idx, (label, fn) in enumerate(all_seven.items(), 1):
    sys_p, usr_p = fn(bench_items[0])
    tpl = usr_p.replace(bench_items[0]["summary"], "{summary}") if bench_items[0]["summary"] in usr_p else usr_p
    if "{summary}" not in tpl:
        tpl = "{summary}\n\nPrice:"
    seven_text.append(f"--- Strategy {idx} ({label}) ---\nSystem: {sys_p}\nUser template: {tpl}")

SUPER_SYSTEM = """You are a prompt engineer. Output valid JSON only: {"system": "...", "user_template": "..."}. The user_template must contain exactly the placeholder {summary}. No markdown, no extra text."""

SUPER_USER = """Below are 7 different prompt strategies for estimating product retail price from a description. Each has a system message and a user message template (with {summary} as placeholder for the product text). Task: create ONE combined "super" prompt that incorporates the best of all seven. Output JSON with keys "system" and "user_template". user_template must use {summary}.

""" + "\n\n".join(seven_text)

SUPER_CACHE_PATH = "super_prompt_cache.json"
REFRESH_SUPER = False

def _parse_super(raw: str):
    data = _extract_json_object(raw)
    if not data or not isinstance(data, dict):
        return None, None
    system = (data.get("system") or "").strip()
    user_tpl = (data.get("user_template") or "").strip().replace("{product}", "{summary}")
    if "{summary}" not in user_tpl:
        user_tpl = "Product:\n{summary}\n\nPrice:"
    return system, user_tpl

if not REFRESH_SUPER and os.path.exists(SUPER_CACHE_PATH):
    try:
        with open(SUPER_CACHE_PATH) as f:
            cached = json.load(f)
        SUPER_SYSTEM_PARSED, SUPER_USER_TEMPLATE = cached.get("system", ""), (cached.get("user_template") or "").replace("{product}", "{summary}")
        if "{summary}" not in SUPER_USER_TEMPLATE:
            SUPER_USER_TEMPLATE = "Product:\n{summary}\n\nPrice:"
        print(f"Loaded super prompt from {SUPER_CACHE_PATH}")
    except Exception as e:
        print(f"Super cache read failed: {e}. Regenerating.")
        SUPER_SYSTEM_PARSED, SUPER_USER_TEMPLATE = None, None
else:
    SUPER_SYSTEM_PARSED, SUPER_USER_TEMPLATE = None, None

if SUPER_SYSTEM_PARSED is None or SUPER_USER_TEMPLATE is None:
    print("Calling LLM to generate super prompt...")
    try:
        raw = _call_for_json(SUPER_SYSTEM, SUPER_USER)
        SUPER_SYSTEM_PARSED, SUPER_USER_TEMPLATE = _parse_super(raw)
        if SUPER_SYSTEM_PARSED is None or SUPER_USER_TEMPLATE is None:
            raise ValueError("Super prompt parse failed")
        with open(SUPER_CACHE_PATH, "w") as f:
            json.dump({"system": SUPER_SYSTEM_PARSED, "user_template": SUPER_USER_TEMPLATE}, f, indent=2)
        print(f"Cached to {SUPER_CACHE_PATH}")
    except Exception as e:
        print(f"Super prompt generation failed: {e}. Using fallback.")
        SUPER_SYSTEM_PARSED = "You are a pricing expert. Consider category, brand, and features. Reply with price only as $XX.XX."
        SUPER_USER_TEMPLATE = "Product:\n{summary}\n\nPrice:"

def format_super(item):
    return SUPER_SYSTEM_PARSED, SUPER_USER_TEMPLATE.replace("{summary}", item["summary"])

print("Super prompt ready.")

## Cell 5 — Run the Ablation Benchmark

In [ ]:
import pandas as pd

def extract_price(text: str) -> float:
    """Extract a numeric price from any LLM reply."""
    text = str(text).replace(",", "").replace("$", "")
    # Look for 'Price: X' pattern first (for CoT format)
    price_match = re.search(r"[Pp]rice[:\s]+([\d.]+)", text)
    if price_match:
        return float(price_match.group(1))
    # Fall back to first number
    match = re.search(r"[-+]?\d*\.?\d+", text)
    return float(match.group()) if match else 0.0

def call_model(system_prompt, user_prompt):
    """Call the configured model with given prompts."""
    return llm_completions(system_prompt, user_prompt, max_tokens=1024)

def benchmark_format(format_fn, label, items):
    """Run format_fn over all items, return MAE."""
    errors = []
    for item in items:
        try:
            sys_p, usr_p = format_fn(item)
            reply  = call_model(sys_p, usr_p)
            pred   = extract_price(reply)
            errors.append(abs(pred - item["price"]))
        except Exception as e:
            print(f"  [{label}] Error: {e}")
    mae = round(sum(errors) / len(errors), 2) if errors else 9999.0
    return mae

ALL_FORMATS = {**FORMATS, **FORMATS_LLM, "Super — Combined": format_super}
ablation_results = {}

for name, fn in ALL_FORMATS.items():
    print(f"Benchmarking format: {name} ...")
    mae = benchmark_format(fn, name, bench_items)
    ablation_results[name] = mae
    print(f"  → MAE = ${mae:.2f}")

best   = min(ablation_results, key=ablation_results.get)
worst  = max(ablation_results, key=ablation_results.get)
spread = ablation_results[worst] - ablation_results[best]

sorted_results = sorted(ablation_results.items(), key=lambda x: x[1])
ranking_df = pd.DataFrame([
    {"Rank": rank, "Format": name, "MAE ($)": f"{mae:.2f}", "vs Best": "Best" if rank == 1 else f"+${mae - sorted_results[0][1]:.2f}"}
    for rank, (name, mae) in enumerate(sorted_results, start=1)
])
print("\n--- Ranking (vs actual price, lower MAE = better) ---")
print(ranking_df.to_string(index=False))

print(f"\n✅ Ablation complete (4 human + 3 LLM + 1 super).")
print(f"   Best format:  {best} (${ablation_results[best]:.2f})")
print(f"   Worst format: {worst} (${ablation_results[worst]:.2f})")
print(f"   Spread: ${spread:.2f} — that's the cost of prompt choice!")

## Cell 6 — Gradio UI: Results + "Try Your Own Prompt"

In [ ]:
import pandas as pd
import gradio as gr

def build_results_df():
    sorted_results = sorted(ablation_results.items(), key=lambda x: x[1])
    best_mae = sorted_results[0][1]
    rows = []
    for rank, (name, mae) in enumerate(sorted_results, start=1):
        rows.append({
            "Rank"    : rank,
            "Format"  : name,
            "MAE ($)" : f"{mae:.2f}",
            "vs Best" : "🏆 Best" if rank == 1 else f"+${mae - best_mae:.2f}",
        })
    return pd.DataFrame(rows)

def build_chart_df():
    return pd.DataFrame([
        {"Format": name, "MAE": mae}
        for name, mae in sorted(ablation_results.items(), key=lambda x: x[1])
    ])

def run_custom_prompt(custom_system: str, custom_user_template: str, progress=gr.Progress()):
    """
    Let user supply their own system prompt and user template.
    Template can use {product} as a placeholder for the product summary.
    """
    if not custom_system.strip():
        return "⚠️ Please enter a system prompt.", None
    if not custom_user_template.strip():
        return "⚠️ Please enter a user message template (use {product} for the product text).", None

    errors = []
    progress(0, desc="Running custom prompt...")

    for i, item in enumerate(bench_items):
        try:
            user_msg = custom_user_template.replace("{product}", item["summary"])
            reply    = call_model(custom_system, user_msg)
            pred     = extract_price(reply)
            errors.append(abs(pred - item["price"]))
        except Exception as e:
            errors.append(None)
        progress((i + 1) / len(bench_items), desc=f"Item {i+1}/{len(bench_items)}")

    errors = [e for e in errors if e is not None]
    if not errors:
        return "❌ All items failed — check your model config.", None

    custom_mae = round(sum(errors) / len(errors), 2)
    best_mae   = min(ablation_results.values())
    diff       = custom_mae - best_mae
    verdict    = "🏆 New best!" if diff < 0 else (f"+${diff:.2f} vs best format" if diff > 0 else "Tied with best!")

    summary = (
        f"**Your prompt MAE: ${custom_mae:.2f}**  ·  {verdict}\n\n"
        f"| Format | MAE |\n|---|---|\n"
    )
    for name, mae in sorted(ablation_results.items(), key=lambda x: x[1]):
        summary += f"| {name} | ${mae:.2f} |\n"
    summary += f"| **Your prompt** | **${custom_mae:.2f}** |\n"

    # Updated chart with custom prompt
    all_results = dict(ablation_results)
    all_results["Your Prompt"] = custom_mae
    chart = pd.DataFrame([
        {"Format": n, "MAE": m}
        for n, m in sorted(all_results.items(), key=lambda x: x[1])
    ])
    return summary, chart


# ── Build the Gradio app ─────────────────────────────────────────────────────
with gr.Blocks(title="Prompt Format Ablation Tester") as demo:

    gr.Markdown(
        "# 🧪 Prompt Format Ablation Tester\n"
        "**How much does prompt wording affect price prediction accuracy?**\n\n"
        f"_Model: `{MODEL_ID}` · Benchmark: {len(bench_items)} Amazon products · Metric: MAE_"
    )

    with gr.Tabs():

        with gr.Tab("📊 Ablation Results"):
            gr.Markdown("### 4 human + 3 LLM + 1 super prompt, ranked by MAE vs actual price (lower = better)")
            gr.Dataframe(value=build_results_df(), interactive=False)
            gr.Markdown("### Visual comparison")
            gr.BarPlot(
                value=build_chart_df(),
                x="Format",
                y="MAE",
                title="MAE by Prompt Format — Lower is Better",
                y_title="MAE ($)",
                height=350,
            )

        with gr.Tab("✍️ Try Your Own Prompt"):
            gr.Markdown(
                "### Write your own prompt and benchmark it against the 8 formats (4 human + 3 LLM + 1 super)\n"
                "`{product}` in your user message will be replaced with the product description."
            )
            with gr.Row():
                custom_system = gr.Textbox(
                    label="System Prompt",
                    value="You are an expert pricing assistant. Estimate the product price and reply with only the dollar amount.",
                    lines=4,
                )
                custom_user = gr.Textbox(
                    label="User Message Template  (use {product} for the product text)",
                    value="Product: {product}\n\nPrice estimate:",
                    lines=4,
                )
            run_btn = gr.Button("🚀 Run Benchmark", variant="primary", size="lg")

            result_md  = gr.Markdown()
            result_chart = gr.BarPlot(
                x="Format",
                y="MAE",
                title="Your Prompt vs Built-in Formats",
                y_title="MAE ($)",
                height=350,
            )

            gr.Examples(
                examples=[
                    ["Respond with a single dollar price. Nothing else.",  "Estimate price: {product}"],
                    ["You are shopping on Amazon. What would you pay for this?", "Item: {product}\nMy budget estimate:"],
                    ["Think like a wholesaler. What's the manufacturer's suggested retail price?", "Product: {product}"],
                ],
                inputs=[custom_system, custom_user],
                label="Example templates to try",
            )

            run_btn.click(
                run_custom_prompt,
                inputs=[custom_system, custom_user],
                outputs=[result_md, result_chart],
            )

demo.launch(inbrowser=True)

---

## Concepts Demonstrated

| Cell | Concept | What it shows |
|---|---|---|
| 3 | HF `datasets` | Load pre-processed dataset from Hub in one line |
| 4 | Prompt engineering | System prompt role, rules, output format — mirrors Week 7 Day 2 `make_prompts()` |
| 5 | Ablation study | Hold model/data constant; vary only the prompt — controlled experiment |
| 5 | Price extraction | Parsing LLM string output including Chain-of-Thought format |
| 6 | Gradio live testing | Interactive custom prompt sandbox with `gr.Progress` feedback |
| 6 | `gr.Examples` | Clickable example templates — mirrors Week 4 Gradio pattern |

---

## Extensions Possible

1. **Swap in your own fine-tuned model** — set `MODEL_ID` to your HF Hub model ID from Week 7
2. **Compare base vs fine-tuned** — run the same 3 formats on both and see if fine-tuning reduces sensitivity to prompt choice
3. **Token efficiency** — track token count per format (shorter format + better accuracy = ideal)
4. **Category breakdown** — filter bench items by category and see if prompt sensitivity varies across product types
5. **Temperature ablation** — add `temperature` as a slider in the UI and see how it interacts with format choice
